In [1]:
import torch 
import torch.nn as nn
import torch.optim as optim
import time
from model import FM_PhysMamba_UNET


In [2]:
small_model = FM_PhysMamba_UNET("small")
large_model = FM_PhysMamba_UNET("large")

In [3]:
try:
    from thop import profile, clever_format
except ImportError:
    print("Please install thop: pip install thop")
    exit()

In [4]:
def measure_peak_memory(func):
    """Helper to measure peak GPU memory usage of a function."""
    torch.cuda.empty_cache()
    torch.cuda.reset_peak_memory_stats()
    
    # Run the function
    func()
    
    peak_bytes = torch.cuda.max_memory_allocated()
    return peak_bytes / (1024**2) # Convert to MB

In [5]:
def benchmark_model(model_name, use_checkpointing):
    print(f"\n--- Benchmarking: {model_name} | Checkpointing: {use_checkpointing} ---")
    
    device = torch.device("cuda" if torch.cuda.is_available() else "cpu")
    if device.type == 'cpu':
        print("You need GPU to run this")
    
    model = FM_PhysMamba_UNET(model_name, gradient_checkpointing = use_checkpointing)
    model.to(device)
    model.train()

    B, C, H, W = 4, 3, 256, 256
    x = torch.randn(B, C, H, W).to(device)
    t = torch.randn(B).to(device)
    optimizer = optim.AdamW(model.parameters(), lr=1e-4)

    # 3. Measurement Setup
    steps = 20
    warmup = 5
    times = []

    # Reset Memory Stats
    torch.cuda.empty_cache()
    torch.cuda.reset_peak_memory_stats()

    print(f"Running {steps} steps (Batch Size {B})...")
    
    for i in range(steps + warmup):
        start_time = time.time()
        
        # --- Standard Training Step ---
        optimizer.zero_grad()
        v_pred, t_map, A_pred = model(x, t)
        loss = v_pred.sum() + t_map.sum() + A_pred.sum()
        loss.backward()
        optimizer.step()
        
        torch.cuda.synchronize() # Wait for GPU to finish
        end_time = time.time()
        
        # Record time only after warmup
        if i >= warmup:
            times.append(end_time - start_time)

    # 4. Results
    avg_time = sum(times) / len(times)
    peak_mem = torch.cuda.max_memory_allocated() / (1024**2) # MB

    return avg_time, peak_mem

In [6]:
if not torch.cuda.is_available():
    print("Error: Benchmarking requires a GPU.")
    exit()

# --- Run Comparisons ---
# 1. STANDARD (High Speed, High RAM)
t_std, mem_std = benchmark_model("small", use_checkpointing=False)

# 2. CHECKPOINTED (Lower Speed, Low RAM)
t_cp, mem_cp = benchmark_model("small", use_checkpointing=True)

# --- Print Summary Table ---
print(f"\n{'='*70}")
print(f"{'METRIC':<20} | {'STANDARD':<15} | {'CHECKPOINTED':<15} | {'CHANGE'}")
print(f"{'-'*70}")

# Memory
mem_diff = mem_cp - mem_std
mem_pct = (mem_diff / mem_std) * 100
print(f"{'Peak VRAM (MB)':<20} | {mem_std:<15.1f} | {mem_cp:<15.1f} | {mem_pct:+.1f}%")

# Speed
time_diff = t_cp - t_std
time_pct = (time_diff / t_std) * 100
print(f"{'Time per Step (s)':<20} | {t_std:<15.4f} | {t_cp:<15.4f} | {time_pct:+.1f}%")

# Throughput
fps_std = 4 / t_std # Batch size 4
fps_cp = 4 / t_cp
print(f"{'Throughput (img/s)':<20} | {fps_std:<15.1f} | {fps_cp:<15.1f} |")
print(f"{'='*70}")


--- Benchmarking: small | Checkpointing: False ---
Running 20 steps (Batch Size 4)...

--- Benchmarking: small | Checkpointing: True ---
Running 20 steps (Batch Size 4)...

METRIC               | STANDARD        | CHECKPOINTED    | CHANGE
----------------------------------------------------------------------
Peak VRAM (MB)       | 5199.7          | 1557.7          | -70.0%
Time per Step (s)    | 0.0577          | 0.0748          | +29.7%
Throughput (img/s)   | 69.4            | 53.5            |
